In [5]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, classification_report
import shap
import matplotlib.pyplot as plt

# Import task 1 splitting module
import splitting as sp

df = pd.read_parquet("prepared_dataset.parquet")

horizons = [1, 2, 3]
results = []

for h in horizons:
    
    for fold_num in range(5):
        fold = sp.get_fold(df, fold=fold_num, horizon=h, task="clf", station_encoding="label")
        
        X_train, y_train = fold["X_train"], fold["y_train"]
        X_test, y_test = fold["X_test"], fold["y_test"]
        class_weight = fold.get("class_weight", "balanced")
        
        # LightGBM Classifier
        lgb_model = lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            class_weight=class_weight,
            random_state=42,
            n_jobs=-1
        )
        lgb_model.fit(X_train, y_train)
        lgb_preds_prob = lgb_model.predict_proba(X_test)[:, 1]
        lgb_preds = (lgb_preds_prob >= 0.5).astype(int)
        
        lgb_auc = roc_auc_score(y_test, lgb_preds_prob)
        lgb_f1 = f1_score(y_test, lgb_preds, zero_division=0)
        
        # Random Forest Classifier
        rf_model = RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            class_weight=class_weight,
            random_state=42,
            n_jobs=-1
        )
        rf_model.fit(X_train, y_train)
        rf_preds_prob = rf_model.predict_proba(X_test)[:, 1]
        rf_preds = (rf_preds_prob >= 0.5).astype(int)
        
        rf_auc = roc_auc_score(y_test, rf_preds_prob)
        rf_f1 = f1_score(y_test, rf_preds, zero_division=0)
        
        results.append({
            "horizon": h,
            "fold": fold_num + 1,
            "lgb_auc": lgb_auc,
            "lgb_f1": lgb_f1,
            "rf_auc": rf_auc,
            "rf_f1": rf_f1
        })
        print(f"Fold {fold_num+1} (h={h}) -> LGB AUC: {lgb_auc:.4f} | RF AUC: {rf_auc:.4f}")

res_df = pd.DataFrame(results)
res_df.to_csv("task4_lgb_rf_metrics.csv", index=False)
print("\nMetrics summary saved to task4_lgb_rf_metrics.csv")

print("\nGenerating SHAP explanations for LightGBM")

fold_shap = sp.get_fold(df, fold=4, horizon=1, task="clf", station_encoding="label")
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(fold_shap["X_test"])

if isinstance(shap_values, list):
    shap_vals_target = shap_values[1]
else:
    shap_vals_target = shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals_target, fold_shap["X_test"], show=False)
plt.title("LightGBM Feature Impact (SHAP Summary Plot) - Horizon 1", pad=20)
plt.tight_layout()
plt.savefig("shap_summary_plot.png", dpi=300)
plt.close()

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals_target, fold_shap["X_test"], plot_type="bar", show=False)
plt.title("LightGBM Global Feature Importance - Horizon 1", pad=20)
plt.tight_layout()
plt.savefig("shap_bar_importance.png", dpi=300)
plt.close()

print("SHAP plots saved successfully: shap_summary_plot.png & shap_bar_importance.png")

[LightGBM] [Info] Number of positive: 437, number of negative: 1613
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005260 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15548
[LightGBM] [Info] Number of data points in the train set: 2050, number of used features: 98
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Fold 1 (h=1) -> LGB AUC: 0.7998 | RF AUC: 0.7938
[LightGBM] [Info] Number of positive: 656, number of negative: 2069
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003129 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15562
[LightGBM] [Info] Number of data points in the train set: 2725, number of used features: 98
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start t

c:\Users\usifa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\shap\explainers\_tree.py:579: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


SHAP plots saved successfully: shap_summary_plot.png & shap_bar_importance.png


In [6]:
import pandas as pd
metrics = pd.read_csv("task4_lgb_rf_metrics.csv")
display(metrics)
print("\nMean Performance per Horizon:")
display(metrics.groupby("horizon")[["lgb_auc", "lgb_f1", "rf_auc", "rf_f1"]].mean())

,horizon,fold,lgb_auc,lgb_f1,rf_auc,rf_f1
0,1,1,0.799754,0.628702,0.793791,0.615385
1,1,2,0.800826,0.460829,0.801006,0.462264
2,1,3,0.846224,0.366667,0.859114,0.432836
3,1,4,0.764896,0.635088,0.717863,0.561837
4,1,5,0.725237,0.227848,0.814797,0.255814
5,2,1,0.743136,0.559471,0.767346,0.566372
6,2,2,0.704907,0.245399,0.726467,0.202703
7,2,3,0.757067,0.209524,0.803343,0.222222
8,2,4,0.682299,0.521257,0.675042,0.547945
9,2,5,0.513330,0.098361,0.712560,0.031250



Mean Performance per Horizon:


,lgb_auc,lgb_f1,rf_auc,rf_f1
horizon,,,,
1,0.787387,0.463827,0.797314,0.465627
2,0.680148,0.326802,0.736952,0.314098
3,0.629996,0.259387,0.709800,0.318894
